# Simple Linear Regression Lab

Experiment with simple linear regression arithmetic by hand and with scikit-learn. This notebook matches the content of the Simple Linear Regression guide exactly, demonstrating how to compute coefficients, evaluate performance measures, analyze error scaling laws, fit polynomial features, and understand joint feature effects.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

## 1. Least Squares Coefficients By Hand

We will fit a line to five points chosen so every intermediate value is exact:

| `x` | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `y` | 2 | 4 | 5 | 4 | 5 |

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 4, 5], dtype=float)
xb, yb = x.mean(), y.mean()

print("1. LEAST SQUARES BY HAND")
print(f"   x̄ = {xb:.1f}   ȳ = {yb:.1f}\n")
print(f"   {'x':>4}{'y':>4}{'(x−x̄)':>9}{'(y−ȳ)':>9}{'(x−x̄)(y−ȳ)':>14}{'(x−x̄)²':>10}")
for xi, yi in zip(x, y):
    print(f"   {xi:>4.0f}{yi:>4.0f}{xi - xb:>9.1f}{yi - yb:>9.1f}"
          f"{(xi - xb) * (yi - yb):>14.1f}{(xi - xb) ** 2:>10.1f}")
Sxy = ((x - xb) * (y - yb)).sum()
Sxx = ((x - xb) ** 2).sum()
print(f"   {'':>17}{'sums →':>18}{Sxy:>14.1f}{Sxx:>10.1f}")

b1 = Sxy / Sxx
b0 = yb - b1 * xb
print(f"\n   b1 = Sxy / Sxx = {Sxy:.0f} / {Sxx:.0f} = {b1}")
print(f"   b0 = ȳ − b1·x̄ = {yb:.0f} − {b1}×{xb:.0f} = {b0}")
print(f"   fitted:  ŷ = {b0} + {b1}·x")

Now let's compute predictions and residuals:

In [ ]:
pred = b0 + b1 * x
resid = y - pred
print(f"\n   {'x':>4}{'y':>4}{'ŷ':>7}{'residual':>11}{'residual²':>12}")
for xi, yi, pi, ri in zip(x, y, pred, resid):
    print(f"   {xi:>4.0f}{yi:>4.0f}{pi:>7.1f}{ri:>11.1f}{ri ** 2:>12.2f}")

m = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"\n   scikit-learn: coef_ = {m.coef_[0]:.4f}   intercept_ = {m.intercept_:.4f}   -> identical")

## 2. The Five Error Measures

All five metrics are summaries of the same residuals. Note that Adjusted $R^2$ must be computed manually since scikit-learn doesn't provide a built-in function for it.

In [ ]:
n, k = len(x), 1
SSE = (resid ** 2).sum()
SST = ((y - yb) ** 2).sum()
r2 = 1 - SSE / SST
print(f"\n2. THE FIVE ERROR MEASURES   (SSE = {SSE:.1f}, SST = {SST:.1f}, n = {n}, k = {k})")
print(f"   {'measure':<14}{'by hand':>10}{'sklearn':>10}   formula")
rows = [
    ("MAE", np.abs(resid).mean(), mean_absolute_error(y, pred), "Σ|r| / n"),
    ("MSE", SSE / n, mean_squared_error(y, pred), "SSE / n"),
    ("RMSE", np.sqrt(SSE / n), root_mean_squared_error(y, pred), "√MSE"),
    ("R²", r2, r2_score(y, pred), "1 − SSE/SST"),
]
for name, hand, sk, formula in rows:
    print(f"   {name:<14}{hand:>10.4f}{sk:>10.4f}   {formula}")
adj = 1 - (1 - r2) * (n - 1) / (n - k - 1)
print(f"   {'Adjusted R²':<14}{adj:>10.4f}{'—':>10}   1 − (1−R²)(n−1)/(n−k−1)  [no sklearn function]")

## 3. Correlation, Slope, and $R^2$ identities

For simple regression with one feature and an intercept, the Pearson correlation $r$, the fitted slope $b_1$, and the coefficient of determination $R^2$ are three views of the exact same physical fact.

In [ ]:
r = np.corrcoef(x, y)[0, 1]
print(f"\n3. r, b1 AND R² ARE THE SAME FACT   (r = {r:.6f})")
for dd in (0, 1):
    print(f"   ddof={dd}:  r × (s_y/s_x) = "
          f"{r * (np.std(y, ddof=dd) / np.std(x, ddof=dd)):.6f}   b1 = {b1:.6f}")
print(f"   r² = {r ** 2:.6f}   R² = {r2:.6f}")

## 4. Error Scaling Law ($1/\sqrt{n}$)

As we collect more data, the standard deviation of our slope estimate shrinks as $1/\sqrt{n}$. Thus, `RMSE * √n` should hold roughly constant. Let's verify this theory by running 3,000 simulations per sample size $n$.

In [ ]:
SIGMA, VAR_X = 2.0, 100 / 12          # x ~ Uniform(0, 10)
predicted = SIGMA / np.sqrt(VAR_X)
print(f"\n4. HOW FAST DOES m̂ APPROACH THE TRUTH   (y = 2.5x + 7, 3000 fits per n)")
print(f"   theory: SD(m̂) = σ/√(n·Var(x)), so RMSE×√n is constant = {predicted:.4f}")
print(f"   {'n':>7}{'RMSE(m̂ − 2.5)':>16}{'RMSE×√n':>10}{'vs theory':>12}")
rmses = []
for nn in [50, 200, 1000, 10000]:
    errs = []
    for seed in range(3000):
        rr = np.random.RandomState(seed + nn * 100003)
        xs = rr.uniform(0, 10, nn)
        ys = 2.5 * xs + 7.0 + rr.normal(0, SIGMA, nn)
        errs.append(LinearRegression().fit(xs.reshape(-1, 1), ys).coef_[0] - 2.5)
    rmse = np.sqrt(np.mean(np.array(errs) ** 2))
    rmses.append(rmse)
    scaled = rmse * np.sqrt(nn)
    print(f"   {nn:>7}{rmse:>16.6f}{scaled:>10.4f}{100 * (scaled - predicted) / predicted:>+11.1f}%")
print("   ratio test:  " + "   ".join(
    f"{rmses[i] / rmses[i + 1]:.3f} (√{[4, 5, 10][i]}={np.sqrt([4, 5, 10][i]):.3f})" for i in range(3)))

## 5. Polynomial expansion for Curves

Linear regression isn't restricted to straight lines in space—it is restricted to lines/planes *linear in its coefficients*. We can easily fit curved shapes (like a parabola) by engineering polynomial columns such as $x^2$.

In [ ]:
rr = np.random.RandomState(50)
xc = rr.uniform(-3, 3, 200)
yc = 2 * xc ** 2 - 3 * xc + 5 + rr.normal(0, 1, 200)   # a parabola
print("\n5. 'LINEAR' MEANS LINEAR IN THE COEFFICIENTS")
for label, F in [("x", xc.reshape(-1, 1)),
                 ("x, x²", np.column_stack([xc, xc ** 2])),
                 ("x, x², x³", np.column_stack([xc, xc ** 2, xc ** 3]))]:
    fit = LinearRegression().fit(F, yc)
    print(f"   given {label:<12} R² = {fit.score(F, yc):.4f}   coefficients "
          f"{np.array2string(fit.coef_, precision=4, floatmode='fixed')}")

## 6. Joint fit vs. Marginal screening

Screening features individually by correlating them with the target is a dangerous anti-pattern. If a dominant feature (like house `size`) explains most of the target variance, a weaker but highly significant feature (like `age`) can appear completely uncorrelated ($r \approx -0.0392$) until we fit them jointly or "partial out" the dominant feature.

In [ ]:
rr = np.random.RandomState(4)
n2 = 120
size = rr.uniform(50, 200, n2)
age = rr.uniform(0, 50, n2)
door = rr.normal(0, 1, n2)                              # front-door colour: pure noise
price = 3000 * size - 800 * age + 50000 + rr.normal(0, 20000, n2)

print("\n6. CORRELATION WITH THE TARGET   (true: price = 3000·size − 800·age + 50000)")
print(f"   {'feature':<14}{'corr':>9}{'fitted b1':>12}{'R² alone':>10}")
for nm, v in [("size", size), ("age", age), ("door_colour", door)]:
    f1 = LinearRegression().fit(v.reshape(-1, 1), price)
    print(f"   {nm:<14}{np.corrcoef(v, price)[0, 1]:>+9.4f}{f1.coef_[0]:>12.2f}"
          f"{f1.score(v.reshape(-1, 1), price):>10.4f}")

resid_price = price - LinearRegression().fit(size.reshape(-1, 1), price).predict(size.reshape(-1, 1))
print("\n   now remove size's contribution from price and look again:")
print(f"   corr(age,  price − size effect) = {np.corrcoef(age, resid_price)[0, 1]:+.4f}")
print(f"   corr(door, price − size effect) = {np.corrcoef(door, resid_price)[0, 1]:+.4f}")
print(f"   slope on age                    = "
      f"{LinearRegression().fit(age.reshape(-1, 1), resid_price).coef_[0]:.2f}   (true −800)")